# AMI Individual Headsets raw baseline


In [1]:
# !pip install -q soundfile pandas pydub transformers accelerate bitsandbytes sentencepiece openai-whisper pyannote.audio python-dotenv


In [2]:
import os
import re
import json
import time
import soundfile as sf
import pandas as pd
import xml.etree.ElementTree as ET

from pathlib import Path
from tqdm import tqdm

def load_env_file(path: str = ".env"):
    try:
        from dotenv import load_dotenv
        load_dotenv(path, override=True)
        return
    except ImportError:
        pass

    if not os.path.exists(path):
        return

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")


load_env_file()

# =========================
# CONFIG AMI
# =========================

AMI_ROOT = Path("/media/user/New Volume/datasets/AMI")

AUDIO_ROOT = AMI_ROOT / "amicorpus"
ANNOT_ROOT = AMI_ROOT / "ami_public_manual_1.6.2"

WORK_ROOT = AMI_ROOT / "work"
CLIP_DIR = WORK_ROOT / "audio_clips"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLIP_DIR.mkdir(parents=True, exist_ok=True)

# pentru test rapid în timp ce se descarcă datasetul
AUDIO_CONDITION = "individual_headset"
OUTPUT_CSV = WORK_ROOT / "ami_manifest_full_individual_headsets_all.csv"

MAX_MEETINGS = None       # toate meeting-urile disponibile
MAX_AUDIO_SECONDS = None  # full audio
CLIP_AUDIO = False        # full audio, fără clipping

GAP_THRESHOLD = 0.8       # unește cuvintele aceluiași speaker dacă pauza <= 0.8s
MIN_FILE_AGE_SECONDS = 60 # evită fișierele care încă se descarcă


# =========================
# HELPERS
# =========================

def local_name(tag: str) -> str:
    """Remove XML namespace."""
    return tag.split("}")[-1] if "}" in tag else tag


def find_words_dir() -> Path:
    candidates = [
        ANNOT_ROOT / "words",
        ANNOT_ROOT / "ami_public_manual_1.6.2" / "words",
        ANNOT_ROOT / "corpusResources" / "words",
    ]

    for c in candidates:
        if c.exists():
            return c

    matches = [p for p in ANNOT_ROOT.rglob("*") if p.is_dir() and p.name == "words"]
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Could not find AMI words directory under: {ANNOT_ROOT}")


def is_audio_ready(audio_path: Path) -> bool:
    """
    Avoid partially downloaded files.
    """
    if not audio_path.exists():
        return False

    age = time.time() - audio_path.stat().st_mtime
    if age < MIN_FILE_AGE_SECONDS:
        return False

    try:
        info = sf.info(str(audio_path))
        return info.frames > 0 and info.samplerate > 0
    except Exception:
        return False


def clip_wav(input_path: Path, output_path: Path, max_seconds: int) -> float:
    """
    Create a short WAV clip from the beginning of a meeting.
    Returns actual duration.
    """
    info = sf.info(str(input_path))
    max_frames = int(max_seconds * info.samplerate)
    frames_to_read = min(info.frames, max_frames)

    with sf.SoundFile(str(input_path), "r") as f:
        audio = f.read(frames=frames_to_read, dtype="float32", always_2d=False)

    sf.write(str(output_path), audio, info.samplerate)
    return frames_to_read / info.samplerate


def clean_word(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text


def count_reference_words(text: str) -> int:
    text = re.sub(r"SPEAKER_\d+:", " ", text)
    return len(text.split())


def parse_word_file(xml_path: Path):
    """
    Parses AMI word-level XML file, e.g. ES2002a.A.words.xml.
    Speaker is inferred from file suffix: A/B/C/D.
    """
    m = re.match(r"(.+?)\.([A-Z])\.words\.xml$", xml_path.name)
    if not m:
        return []

    meeting_id = m.group(1)
    speaker_raw = m.group(2)

    try:
        tree = ET.parse(xml_path)
    except Exception as e:
        print(f"[WARN] Could not parse {xml_path}: {e}")
        return []

    root = tree.getroot()
    words = []

    for elem in root.iter():
        if local_name(elem.tag) != "w":
            continue

        text = clean_word("".join(elem.itertext()))
        if not text:
            continue

        start = elem.attrib.get("starttime")
        end = elem.attrib.get("endtime")

        if start is None or end is None:
            continue

        try:
            start = float(start)
            end = float(end)
        except ValueError:
            continue

        if end <= start:
            continue

        words.append({
            "meeting_id": meeting_id,
            "speaker_raw": speaker_raw,
            "start": start,
            "end": end,
            "word": text,
        })

    return words


def build_reference_from_words(words, max_time=None):
    """
    Builds:
    - speaker-attributed reference transcript
    - speaker segment reference for DER/JER
    from AMI word-level timing annotations.
    """
    if max_time is not None:
        filtered = []
        for w in words:
            if w["start"] >= max_time:
                continue

            w = dict(w)
            w["end"] = min(w["end"], max_time)

            if w["end"] > w["start"]:
                filtered.append(w)

        words = filtered

    words = sorted(words, key=lambda x: (x["start"], x["end"]))

    speaker_map = {}
    segments = []
    current = None

    for w in words:
        raw_speaker = w["speaker_raw"]

        if raw_speaker not in speaker_map:
            speaker_map[raw_speaker] = f"SPEAKER_{len(speaker_map):02d}"

        speaker = speaker_map[raw_speaker]

        if current is None:
            current = {
                "start": w["start"],
                "end": w["end"],
                "speaker": speaker,
                "words": [w["word"]],
            }
            continue

        same_speaker = current["speaker"] == speaker
        small_gap = (w["start"] - current["end"]) <= GAP_THRESHOLD

        if same_speaker and small_gap:
            current["end"] = max(current["end"], w["end"])
            current["words"].append(w["word"])
        else:
            segments.append(current)
            current = {
                "start": w["start"],
                "end": w["end"],
                "speaker": speaker,
                "words": [w["word"]],
            }

    if current is not None:
        segments.append(current)

    final_segments = []
    transcript_lines = []

    for s in segments:
        text = " ".join(s["words"]).strip()
        if not text:
            continue

        item = {
            "start": float(s["start"]),
            "end": float(s["end"]),
            "speaker": s["speaker"],
            "text": text,
        }

        final_segments.append(item)
        transcript_lines.append(f'{s["speaker"]}: {text}')

    text_reference = "\n".join(transcript_lines)
    return text_reference, final_segments


# =========================
# BUILD AMI MANIFEST
# =========================

words_dir = find_words_dir()
print(f"Using AMI words dir: {words_dir}")

audio_files = sorted(AUDIO_ROOT.glob("*/audio/*.Headset-*.wav"))
print(f"Found Mix-Headset candidates: {len(audio_files)}")

rows = []

for audio_path in tqdm(audio_files):
    meeting_id = audio_path.parent.parent.name

    if not is_audio_ready(audio_path):
        print(f"[SKIP] Audio not ready yet: {meeting_id}")
        continue

    word_files = sorted(words_dir.rglob(f"{meeting_id}.*.words.xml"))

    if not word_files:
        print(f"[WARN] No word XML files for {meeting_id}")
        continue

    all_words = []
    for wf in word_files:
        all_words.extend(parse_word_file(wf))

    if not all_words:
        print(f"[WARN] No parsed words for {meeting_id}")
        continue

    if CLIP_AUDIO and MAX_AUDIO_SECONDS is not None:
        clip_path = CLIP_DIR / f"{meeting_id}_first_{MAX_AUDIO_SECONDS}s.wav"

        if not clip_path.exists():
            actual_duration = clip_wav(audio_path, clip_path, MAX_AUDIO_SECONDS)
        else:
            actual_duration = min(MAX_AUDIO_SECONDS, sf.info(str(clip_path)).duration)

        used_audio_path = clip_path
        reference_limit = actual_duration
    else:
        used_audio_path = audio_path
        reference_limit = None

    text_reference, segments = build_reference_from_words(
        all_words,
        max_time=reference_limit
    )

    if not segments:
        print(f"[WARN] Empty reference after filtering for {meeting_id}")
        continue

    num_speakers = len(set(s["speaker"] for s in segments))
    num_reference_words = count_reference_words(text_reference)
    reference_start = min(s["start"] for s in segments)
    reference_end = max(s["end"] for s in segments)

    rows.append({
        "audio_number": audio_path.stem,
        "meeting_id": meeting_id,
        "audio_condition": AUDIO_CONDITION,
        "audio_path": str(used_audio_path),
        "text_reference": text_reference,
        "speaker_segments_reference": json.dumps(segments, ensure_ascii=False),
        "num_speakers": num_speakers,
        "num_reference_segments": len(segments),
        "num_reference_words": num_reference_words,
        "reference_start": reference_start,
        "reference_end": reference_end,
        "source_audio_path": str(audio_path),
    })

    print(
        f"[OK] {meeting_id}: "
        f"speakers={num_speakers}, "
        f"segments={len(segments)}, "
        f"words={num_reference_words}, "
        f"audio={used_audio_path.name}"
    )

    if MAX_MEETINGS is not None and len(rows) >= MAX_MEETINGS:
        break

df_final = pd.DataFrame(rows)

if df_final.empty:
    raise RuntimeError(
        "No AMI meetings were added to the manifest. "
        "Check whether Mix-Headset WAV files are downloaded and whether words XML files exist."
    )

df_final.to_csv(OUTPUT_CSV, index=False)

print()
print(df_final[[
    "audio_number",
    "audio_path",
    "num_speakers",
    "num_reference_segments",
    "num_reference_words",
    "reference_start",
    "reference_end",
]])
print(f"\nSaved AMI manifest to: {OUTPUT_CSV}")

Using AMI words dir: /media/user/New Volume/datasets/AMI/ami_public_manual_1.6.2/words
Found Mix-Headset candidates: 575


  0%|          | 2/575 [00:00<00:45, 12.59it/s]

[OK] EN2001a: speakers=5, segments=2123, words=16093, audio=EN2001a.Headset-0.wav
[OK] EN2001a: speakers=5, segments=2123, words=16093, audio=EN2001a.Headset-1.wav
[OK] EN2001a: speakers=5, segments=2123, words=16093, audio=EN2001a.Headset-2.wav


  1%|          | 4/575 [00:00<00:44, 12.82it/s]

[OK] EN2001a: speakers=5, segments=2123, words=16093, audio=EN2001a.Headset-3.wav
[OK] EN2001a: speakers=5, segments=2123, words=16093, audio=EN2001a.Headset-4.wav
[OK] EN2001b: speakers=4, segments=1353, words=9939, audio=EN2001b.Headset-0.wav


...

[OK] TS3012b: speakers=4, segments=2032, words=8477, audio=TS3012b.Headset-0.wav
[OK] TS3012b: speakers=4, segments=2032, words=8477, audio=TS3012b.Headset-1.wav
[OK] TS3012b: speakers=4, segments=2032, words=8477, audio=TS3012b.Headset-2.wav
[OK] TS3012b: speakers=4, segments=2032, words=8477, audio=TS3012b.Headset-3.wav
[OK] TS3012c: speakers=4, segments=2272, words=8277, audio=TS3012c.Headset-0.wav


 99%|█████████▉| 572/575 [00:15<00:00, 27.52it/s]

[OK] TS3012c: speakers=4, segments=2272, words=8277, audio=TS3012c.Headset-1.wav
[OK] TS3012c: speakers=4, segments=2272, words=8277, audio=TS3012c.Headset-2.wav
[OK] TS3012c: speakers=4, segments=2272, words=8277, audio=TS3012c.Headset-3.wav
[OK] TS3012d: speakers=4, segments=1818, words=6732, audio=TS3012d.Headset-0.wav
[OK] TS3012d: speakers=4, segments=1818, words=6732, audio=TS3012d.Headset-1.wav


100%|██████████| 575/575 [00:15<00:00, 36.05it/s]


[OK] TS3012d: speakers=4, segments=1818, words=6732, audio=TS3012d.Headset-2.wav
[OK] TS3012d: speakers=4, segments=1818, words=6732, audio=TS3012d.Headset-3.wav

          audio_number                                         audio_path  \
0    EN2001a.Headset-0  /media/user/New Volume/datasets/AMI/amicorpus/...   
1    EN2001a.Headset-1  /media/user/New Volume/datasets/AMI/amicorpus/...   
2    EN2001a.Headset-2  /media/user/New Volume/datasets/AMI/amicorpus/...   
3    EN2001a.Headset-3  /media/user/New Volume/datasets/AMI/amicorpus/...   
4    EN2001a.Headset-4  /media/user/New Volume/datasets/AMI/amicorpus/...   
..                 ...                                                ...   
570  TS3012c.Headset-3  /media/user/New Volume/datasets/AMI/amicorpus/...   
571  TS3012d.Headset-0  /media/user/New Volume/datasets/AMI/amicorpus/...   
572  TS3012d.Headset-1  /media/user/New Volume/datasets/AMI/amicorpus/...   
573  TS3012d.Headset-2  /media/user/New Volume/datasets/AMI/amicorp

In [3]:
df_manifest = pd.read_csv(WORK_ROOT / "ami_manifest_full_individual_headsets_all.csv")
print(len(df_manifest))
print(df_manifest[["audio_number", "audio_path", "num_speakers", "num_reference_words"]].head())


575
        audio_number                                         audio_path  \
0  EN2001a.Headset-0  /media/user/New Volume/datasets/AMI/amicorpus/...   
1  EN2001a.Headset-1  /media/user/New Volume/datasets/AMI/amicorpus/...   
2  EN2001a.Headset-2  /media/user/New Volume/datasets/AMI/amicorpus/...   
3  EN2001a.Headset-3  /media/user/New Volume/datasets/AMI/amicorpus/...   
4  EN2001a.Headset-4  /media/user/New Volume/datasets/AMI/amicorpus/...   

   num_speakers  num_reference_words  
0             5                16093  
1             5                16093  
2             5                16093  
3             5                16093  
4             5                16093  


# Config

In [4]:
import time
start_time = time.time()

import os
import re
import json
import tempfile
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from itertools import permutations

from jiwer import wer
from rapidfuzz.distance import Levenshtein

import pandas as pd
import soundfile as sf
import torch
import whisper

from pydub import AudioSegment
from pyannote.audio import Pipeline
from pyannote.core import Annotation, Segment
from pyannote.metrics.diarization import (
    DiarizationErrorRate,
    JaccardErrorRate,
    DiarizationPurity,
    DiarizationCoverage,
)
from whisper.audio import log_mel_spectrogram, pad_or_trim, N_FRAMES
from tqdm import tqdm

def load_env_file(path: str = ".env"):
    try:
        from dotenv import load_dotenv
        load_dotenv(path, override=True)
        return
    except ImportError:
        pass

    if not os.path.exists(path):
        return

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")


load_env_file()


AMI_ROOT = "/media/user/New Volume/datasets/AMI"
WORK_ROOT = f"{AMI_ROOT}/work"

RUN_NAME = "ami_full_individual_headsets_all_base_en"

INPUT_CSV = f"{WORK_ROOT}/ami_manifest_full_individual_headsets_all.csv"

RAW_OUTPUT_CSV = f"{WORK_ROOT}/raw_pipeline_outputs_{RUN_NAME}.csv"
METRICS_OUTPUT_CSV = f"{WORK_ROOT}/results_with_metrics_{RUN_NAME}.csv"
SUMMARY_TEXT_CSV = f"{WORK_ROOT}/summary_text_metrics_{RUN_NAME}.csv"
SUMMARY_SPEAKER_CSV = f"{WORK_ROOT}/summary_speaker_metrics_{RUN_NAME}.csv"

# AMI are în general meeting-uri multi-speaker.
# Folosim num_speakers din CSV, extras din ground truth.
NUM_SPEAKERS = None

# model local mic pentru ASR; poți schimba prin variabila de mediu WHISPER_MODEL_NAME
WHISPER_MODEL_NAME = os.getenv("WHISPER_MODEL_NAME", "small")

PYANNOTE_MODEL_NAME = os.getenv("PYANNOTE_MODEL_NAME", "pyannote/speaker-diarization-3.1")

LANGUAGE = "en"

/home/user/anaconda3/envs/fgcs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## LLM helper

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


LOCAL_LLM_MODEL_NAME = os.getenv("LOCAL_LLM_MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
LOCAL_LLM_VARIANT_PREFIX = re.sub(r"[^A-Za-z0-9]+", "_", LOCAL_LLM_MODEL_NAME.split("/")[-1]).strip("_").lower()
RUN_LOCAL_LLM = os.getenv("RUN_LOCAL_LLM", "False").lower() in {"1", "true", "yes", "on"}

# Alternative locale mici:
# LOCAL_LLM_MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# LOCAL_LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
# LOCAL_LLM_MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
# LOCAL_LLM_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"


class LocalLLMPostProcessor:
    def __init__(self, model_name: str = LOCAL_LLM_MODEL_NAME):
        self.model_name = model_name

        use_cuda = torch.cuda.is_available()
        quant_config = None
        torch_dtype = torch.float16 if use_cuda else torch.float32

        if use_cuda:
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
        )

        model_kwargs = {
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
        }
        if use_cuda:
            model_kwargs.update({
                "device_map": "auto",
                "quantization_config": quant_config,
            })
        else:
            model_kwargs.update({"torch_dtype": torch_dtype})

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            **model_kwargs,
        )
        if not use_cuda:
            self.model.to("cpu")

    def create_prompt(self, text: str, shot_type: str = "zero_shot") -> str:
        base_instruction = (
            "You are given a speaker-tagged transcript with labels such as SPEAKER_00, "
            "SPEAKER_01, SPEAKER_02, and so on.\n"
            "Your task is only to correct speaker attribution.\n"
            "Do not add, remove, paraphrase, or replace any word.\n"
            "Only move existing words or spans between the existing speakers when necessary.\n"
            "Do not invent new speakers.\n"
            "Return only the corrected transcript."
        )

        one_shot_example = """
Input:
SPEAKER_00: How are you doing today? I
SPEAKER_01: am doing very well. How was everything at the
SPEAKER_00: party? Oh, the party? It was awesome. We had lots of fun. Good
SPEAKER_01: to hear!

Output:
SPEAKER_00: How are you doing today?
SPEAKER_01: I am doing very well. How was everything at the party?
SPEAKER_00: Oh, the party? It was awesome. We had lots of fun.
SPEAKER_01: Good to hear!
""".strip()

        if shot_type == "one_shot":
            user_prompt = f"{base_instruction}\n\n{one_shot_example}\n\nNow correct:\n{text}"
        else:
            user_prompt = f"{base_instruction}\n\nCorrect this transcript:\n{text}"

        return user_prompt

    def get_completion(self, text: str, shot_type: str = "zero_shot", temperature: float = 0.0) -> str:
        prompt = self.create_prompt(text, shot_type)

        messages = [
            {"role": "system", "content": "Follow the formatting constraints exactly."},
            {"role": "user", "content": prompt},
        ]

        if hasattr(self.tokenizer, "apply_chat_template") and self.tokenizer.chat_template:
            input_text = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            input_text = f"System: {messages[0]['content']}\nUser: {messages[1]['content']}\nAssistant:"

        inputs = self.tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=12000,
        ).to(self.model.device)

        generation_kwargs = {
            **inputs,
            "max_new_tokens": 4096,
            "do_sample": temperature > 0,
            "pad_token_id": self.tokenizer.eos_token_id,
        }

        if temperature > 0:
            generation_kwargs["temperature"] = temperature

        with torch.no_grad():
            output_ids = self.model.generate(**generation_kwargs)

        generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
        output = self.tokenizer.decode(generated_ids, skip_special_tokens=True)

        return output.strip()


# Pyannote class

In [6]:
class PyannoteProcessor:
    """
    Performs speaker diarization using Pyannote.
    Avoids pyannote's internal AudioDecoder by loading audio with soundfile.
    """

    def __init__(self, model_name: str = PYANNOTE_MODEL_NAME, num_speakers: Optional[int] = NUM_SPEAKERS):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        token = os.getenv("HUGGINGFACE_TOKEN")

        if token is None:
            raise ValueError("Missing HUGGINGFACE_TOKEN environment variable.")

        try:
            try:
                self.pipeline = Pipeline.from_pretrained(model_name, token=token)
            except TypeError:
                self.pipeline = Pipeline.from_pretrained(model_name, use_auth_token=token)
        except Exception as exc:
            error_text = str(exc)
            error_type = type(exc).__name__
            if "403" in error_text or "GatedRepo" in error_type or "gated repo" in error_text.lower():
                raise RuntimeError(
                    "Pyannote model access is gated for this HuggingFace token. "
                    "Accept access for pyannote/speaker-diarization-3.1 and pyannote/segmentation-3.0."
                ) from exc
            raise

        self.pipeline.to(self.device)
        self.default_num_speakers = num_speakers

    @staticmethod
    def load_audio_for_pyannote(audio_file_path: str):
        audio, sample_rate = sf.read(audio_file_path, dtype="float32", always_2d=True)

        # soundfile returns shape: [samples, channels]
        # pyannote expects: [channels, samples]
        waveform = audio.T

        # If stereo/multichannel, convert to mono for diarization.
        if waveform.shape[0] > 1:
            waveform = waveform.mean(axis=0, keepdims=True)

        waveform = torch.from_numpy(waveform)

        return {
            "waveform": waveform,
            "sample_rate": sample_rate,
        }

    @staticmethod
    def unwrap_diarization(diarization):
        if hasattr(diarization, "itertracks"):
            return diarization

        for attr in ("speaker_diarization", "diarization", "annotation"):
            value = getattr(diarization, attr, None)
            if hasattr(value, "itertracks"):
                return value

        if isinstance(diarization, dict):
            for key in ("speaker_diarization", "diarization", "annotation"):
                value = diarization.get(key)
                if hasattr(value, "itertracks"):
                    return value

        available = sorted(name for name in dir(diarization) if not name.startswith("_"))
        raise TypeError(
            f"Unsupported pyannote diarization output type: {type(diarization).__name__}. "
            f"Could not find an Annotation-like object with itertracks(). "
            f"Available attributes: {available[:30]}"
        )

    def perform_diarization(self, audio_file_path: str, num_speakers: Optional[int] = None):
        n = num_speakers if num_speakers is not None else self.default_num_speakers

        audio_input = self.load_audio_for_pyannote(audio_file_path)

        if n is None or pd.isna(n):
            diarization = self.pipeline(audio_input)
        else:
            diarization = self.pipeline(audio_input, num_speakers=int(n))

        return self.unwrap_diarization(diarization)

    @staticmethod
    def save_rttm(diarization, output_rttm_path: str):
        diarization = PyannoteProcessor.unwrap_diarization(diarization)
        with open(output_rttm_path, "w", encoding="utf-8") as f:
            diarization.write_rttm(f)

    @staticmethod
    def rttm_to_dataframe(rttm_file_path: str) -> pd.DataFrame:
        columns = [
            "type", "file_id", "channel", "start_time", "duration",
            "orthography", "confidence", "speaker", "x", "y"
        ]
        rows = []

        with open(rttm_file_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 10:
                    rows.append(parts)

        df = pd.DataFrame(rows, columns=columns)
        df["start_time"] = df["start_time"].astype(float)
        df["duration"] = df["duration"].astype(float)
        df["end_time"] = df["start_time"] + df["duration"]

        return df[["file_id", "channel", "start_time", "duration", "end_time", "speaker"]]


# Whisper class

In [7]:
class WhisperProcessor:
    def __init__(self, model_name: str = WHISPER_MODEL_NAME):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = whisper.load_model(model_name).to(self.device)

    def transcribe_audio(self, audio_file: str, language: Optional[str] = None) -> dict:
        result = self.model.transcribe(
            audio_file,
            language=language,
            fp16=(self.device == "cuda"),
            temperature=0.0,
            condition_on_previous_text=False,
            verbose=False,
        )
        return result

    def detect_audio_language(self, audio_array) -> Tuple[str, float]:
        mel_segment = pad_or_trim(log_mel_spectrogram(audio_array), N_FRAMES).to(self.model.device)
        _, probs = self.model.detect_language(mel_segment)
        detected_language = max(probs, key=probs.get)
        return detected_language, probs[detected_language]

    @staticmethod
    def transcribe_segment_file(model, segment_path: str, language: Optional[str] = None) -> str:
        result = model.transcribe(
            segment_path,
            language=language,
            temperature=0.0,
            condition_on_previous_text=False,
            verbose=False,
        )
        return result["text"].strip()

# Local LLM class


In [8]:
# Cloud post-processing was removed.
# The pipeline uses LocalLLMPostProcessor from the LLM helper cell above.


# Data structures and pipeline

In [9]:
@dataclass
class SegmentRecord:
    start: float
    end: float
    speaker: str
    text: str


class AudioProcessor:
    def __init__(self, diarizer: PyannoteProcessor, asr: WhisperProcessor):
        self.diarizer = diarizer
        self.asr = asr

    def diarization_to_segments(self, diarization) -> List[Tuple[float, float, str]]:
        segments = []
        for turn, _, speaker in diarization.itertracks(yield_label=True):
            segments.append((float(turn.start), float(turn.end), str(speaker)))
        return segments

    def transcribe_diarized_segments(
        self,
        audio_path: str,
        diarization,
        language: Optional[str] = None,
        reference_start: Optional[float] = None,
        reference_end: Optional[float] = None,
    ) -> List[SegmentRecord]:
        audio = AudioSegment.from_file(audio_path)
        records = []

        for start, end, speaker in self.diarization_to_segments(diarization):
            if reference_start is not None and end <= reference_start:
                continue
            if reference_end is not None and start >= reference_end:
                continue

            if reference_start is not None:
                start = max(start, reference_start)
            if reference_end is not None:
                end = min(end, reference_end)

            if end <= start:
                continue

            start_ms = int(start * 1000)
            end_ms = int(end * 1000)
            segment_audio = audio[start_ms:end_ms]

            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
                temp_path = tmp.name

            try:
                segment_audio.export(temp_path, format="wav")
                text = self.asr.transcribe_audio(temp_path, language=language)["text"].strip()
                records.append(SegmentRecord(start, end, speaker, text))
            finally:
                if os.path.exists(temp_path):
                    os.remove(temp_path)

        return records

    @staticmethod
    def normalize_speaker_name(raw_speaker: str, mapping: Dict[str, str]) -> str:
        if raw_speaker not in mapping:
            mapping[raw_speaker] = f"SPEAKER_{len(mapping):02d}"
        return mapping[raw_speaker]

    @staticmethod
    def build_transcript(records: List[SegmentRecord]) -> str:
        lines = []
        speaker_map = {}

        for r in records:
            normalized = AudioProcessor.normalize_speaker_name(r.speaker, speaker_map)
            lines.append(f"{normalized}: {r.text}")

        return "\n".join(lines)

    @staticmethod
    def build_serializable_segments(records: List[SegmentRecord]) -> List[Dict]:
        speaker_map = {}
        serialized = []

        for r in records:
            normalized = AudioProcessor.normalize_speaker_name(r.speaker, speaker_map)
            serialized.append({
                "start": r.start,
                "end": r.end,
                "speaker": normalized,
                "text": r.text,
            })

        return serialized

# Serialization helpers

In [10]:
def serialize_segment_records(records: List[Dict]) -> str:
    return json.dumps(records)

def deserialize_segment_records(serialized: str) -> List[Tuple[float, float, str]]:
    if pd.isna(serialized) or not serialized:
        return []

    items = json.loads(serialized)
    return [
        (float(item["start"]), float(item["end"]), str(item["speaker"]))
        for item in items
    ]

def is_valid_speaker_transcript(text: str) -> bool:
    return isinstance(text, str) and bool(re.search(r"SPEAKER_\d+:", text))

def token_list(text: str):
    text = re.sub(r"SPEAKER_\d+:", " ", text)
    text = text.lower()
    return re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", text)


def normalize_transcript_for_lexical_check(text: str) -> List[str]:
    text = re.sub(r"SPEAKER_\d+:", " ", text)
    text = text.lower()
    return re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", text)

def lexical_content_preserved(source: str, candidate: str) -> bool:
    return normalize_transcript_for_lexical_check(source) == normalize_transcript_for_lexical_check(candidate)


def split_transcript_lines(transcript: str, max_lines: int = 40):
    lines = [line for line in transcript.splitlines() if line.strip()]
    chunks = []

    for i in range(0, len(lines), max_lines):
        chunks.append("\n".join(lines[i:i + max_lines]))

    return chunks


def correct_transcript_in_chunks(llm_model, transcript: str, shot_type: str = "zero_shot", max_lines: int = 40):
    chunks = split_transcript_lines(transcript, max_lines=max_lines)
    corrected_chunks = []

    for i, chunk in enumerate(tqdm(chunks, desc=f"LLM {shot_type} chunks")):
        corrected = llm_model.get_completion(
            text=chunk,
            shot_type=shot_type,
            temperature=0.0,
        )

        if not is_valid_speaker_transcript(corrected):
            print(f"[WARN] Invalid chunk {i}; keeping original.")
            corrected = chunk

        if not lexical_content_preserved(chunk, corrected):
            print(f"[WARN] Lexical content changed in chunk {i}; keeping original.")
            corrected = chunk

        corrected_chunks.append(corrected)

    return "\n".join(corrected_chunks)


# Metrics

## Text metrics

In [11]:
from typing import List, Tuple, Dict
from itertools import permutations
from jiwer import wer
from rapidfuzz.distance import Levenshtein

def remove_speaker_tags(text: str) -> str:
    text = re.sub(r"SPEAKER_\d{2}:", " ", str(text))
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def levenshtein_counts(ref_words: List[str], hyp_words: List[str]) -> Tuple[int, int]:
    edits = Levenshtein.distance(ref_words, hyp_words)
    return edits, len(ref_words)

def compute_wer(ref_text: str, hyp_text: str) -> float:
    ref = remove_speaker_tags(ref_text)
    hyp = remove_speaker_tags(hyp_text)

    if not ref:
        return 0.0

    return wer(ref, hyp)

def parse_speaker_transcript(transcript: str) -> Dict[str, List[str]]:
    pattern = r"(SPEAKER_\d+):"
    parts = re.split(pattern, transcript)
    speaker_to_text = {}

    for i in range(1, len(parts), 2):
        speaker = parts[i]
        text = parts[i + 1].strip()
        speaker_to_text.setdefault(speaker, []).append(text)

    return {spk: " ".join(chunks).strip() for spk, chunks in speaker_to_text.items()}


def compute_cpwer(reference_transcript: str, hypothesis_transcript: str) -> float:
    ref_by_spk = parse_speaker_transcript(reference_transcript)
    hyp_by_spk = parse_speaker_transcript(hypothesis_transcript)

    ref_speakers = sorted(ref_by_spk.keys())
    hyp_speakers = sorted(hyp_by_spk.keys())

    if len(ref_speakers) != len(hyp_speakers):
        return float("nan")

    best = float("inf")

    for perm in permutations(hyp_speakers):
        total_edits = 0
        total_words = 0

        for ref_spk, hyp_spk in zip(ref_speakers, perm):
            edits, nref = levenshtein_counts(
                token_list(ref_by_spk[ref_spk]),
                token_list(hyp_by_spk[hyp_spk]),
            )
            total_edits += edits
            total_words += nref

        score = total_edits / total_words if total_words > 0 else 0.0
        best = min(best, score)

    return best



## Speaker metrics

In [12]:
class SpeakerMetrics:
    @staticmethod
    def segments_to_annotation(segments: List[Tuple[float, float, str]]) -> Annotation:
        ann = Annotation()
        for start, end, speaker in segments:
            ann[Segment(start, end)] = speaker
        return ann

    @staticmethod
    def compute_all(reference_segments, hypothesis_segments) -> Dict[str, float]:
        ref_ann = SpeakerMetrics.segments_to_annotation(reference_segments)
        hyp_ann = SpeakerMetrics.segments_to_annotation(hypothesis_segments)

        der_metric = DiarizationErrorRate(collar=0.0, skip_overlap=False)
        jer_metric = JaccardErrorRate(collar=0.0)
        purity_metric = DiarizationPurity()
        coverage_metric = DiarizationCoverage()

        return {
            "DER": der_metric(ref_ann, hyp_ann),
            "JER": jer_metric(ref_ann, hyp_ann),
            "Purity": purity_metric(ref_ann, hyp_ann),
            "Coverage": coverage_metric(ref_ann, hyp_ann),
        }

# Core pipeline

In [13]:
def process_one_audio(
    audio_number,
    audio_path,
    audio_processor,
    llm_variants,
    num_speakers: Optional[int] = None,
    language: Optional[str] = LANGUAGE,
    reference_start: Optional[float] = None,
    reference_end: Optional[float] = None,
):
    row = {
        "audio_number": audio_number,
        "audio_path": audio_path,
        "status": "ok",
    }

    t0 = time.time()

    diarization = audio_processor.diarizer.perform_diarization(
        audio_file_path=audio_path,
        num_speakers=num_speakers,
    )

    segment_records = audio_processor.transcribe_diarized_segments(
        audio_path=audio_path,
        diarization=diarization,
        language=language,
        reference_start=reference_start,
        reference_end=reference_end,
    )

    baseline_transcript = audio_processor.build_transcript(segment_records)
    serialized_segments = AudioProcessor.build_serializable_segments(segment_records)

    row["transcript_without_LM"] = baseline_transcript
    row["speaker_segments_without_LM"] = serialize_segment_records(serialized_segments)

    for variant_name, (llm_model, shot_type) in llm_variants.items():
        corrected = llm_model.get_completion(
            text=baseline_transcript,
            shot_type=shot_type,
            temperature=0.0,
        )

        if not is_valid_speaker_transcript(corrected):
            corrected = baseline_transcript

        if not lexical_content_preserved(baseline_transcript, corrected):
            corrected = baseline_transcript

        row[f"transcript_{variant_name}"] = corrected

    row["runtime_seconds"] = time.time() - t0
    return row


def run_pipeline_on_dataset(df, audio_processor, llm_variants, output_csv: Optional[str] = None, resume: bool = True):
    all_rows = []
    done_audio_numbers = set()

    if resume and output_csv is not None and os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        all_rows = existing_df.to_dict("records")

        if "audio_number" in existing_df.columns:
            done_audio_numbers = set(existing_df["audio_number"].astype(str))

        print(f"[RESUME] Found {len(done_audio_numbers)} already processed meetings.")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        audio_number = str(row["audio_number"])

        if audio_number in done_audio_numbers:
            continue

        audio_path = row["audio_path"]

        num_speakers = None
        if "num_speakers" in row and not pd.isna(row["num_speakers"]):
            num_speakers = int(row["num_speakers"])

        reference_start = None
        reference_end = None

        if "reference_start" in row and not pd.isna(row["reference_start"]):
            reference_start = float(row["reference_start"])

        if "reference_end" in row and not pd.isna(row["reference_end"]):
            reference_end = float(row["reference_end"])

        try:
            result_row = process_one_audio(
                audio_number=audio_number,
                audio_path=audio_path,
                audio_processor=audio_processor,
                llm_variants=llm_variants,
                num_speakers=num_speakers,
                language=LANGUAGE,
                reference_start=reference_start,
                reference_end=reference_end,
            )

            for col in [
                "text_reference",
                "speaker_segments_reference",
                "num_speakers",
                "reference_start",
                "reference_end",
            ]:
                if col in row:
                    result_row[col] = row[col]

            all_rows.append(result_row)

        except Exception as e:
            print(f"[ERROR] audio_number={audio_number}, path={audio_path}, error={e}")
            all_rows.append({
                "audio_number": audio_number,
                "audio_path": audio_path,
                "status": "failed",
                "error": str(e),
            })

        if output_csv is not None:
            pd.DataFrame(all_rows).to_csv(output_csv, index=False)

    return pd.DataFrame(all_rows)


# Evaluation

In [14]:
def evaluate_all_methods(df: pd.DataFrame) -> pd.DataFrame:
    methods = [column.replace("transcript_", "") for column in df.columns if column.startswith("transcript_")]

    for method in methods:
        wer_scores = []
        cpwer_scores = []

        for _, row in df.iterrows():
            if row.get("status") == "failed":
                wer_scores.append(float("nan"))
                cpwer_scores.append(float("nan"))
                continue

            ref = row.get("text_reference", None)
            hyp = row.get(f"transcript_{method}", None)

            if pd.isna(ref) or pd.isna(hyp) or ref is None or hyp is None:
                wer_scores.append(float("nan"))
                cpwer_scores.append(float("nan"))
            else:
                wer_scores.append(compute_wer(ref, hyp))
                cpwer_scores.append(compute_cpwer(ref, hyp))

        df[f"WER_{method}"] = wer_scores
        df[f"cpWER_{method}"] = cpwer_scores

    # speaker metrics doar pentru baseline diarization
    ders, jers, purities, coverages = [], [], [], []

    for _, row in df.iterrows():
        if row.get("status") == "failed":
            ders.append(float("nan"))
            jers.append(float("nan"))
            purities.append(float("nan"))
            coverages.append(float("nan"))
            continue

        ref_segments_raw = row.get("speaker_segments_reference", None)
        hyp_segments_raw = row.get("speaker_segments_without_LM", None)

        if pd.isna(ref_segments_raw) or pd.isna(hyp_segments_raw) or ref_segments_raw is None or hyp_segments_raw is None:
            ders.append(float("nan"))
            jers.append(float("nan"))
            purities.append(float("nan"))
            coverages.append(float("nan"))
            continue

        ref_segments = deserialize_segment_records(ref_segments_raw)
        hyp_segments = deserialize_segment_records(hyp_segments_raw)

        metrics = SpeakerMetrics.compute_all(ref_segments, hyp_segments)
        ders.append(metrics["DER"])
        jers.append(metrics["JER"])
        purities.append(metrics["Purity"])
        coverages.append(metrics["Coverage"])

    df["DER_without_LM"] = ders
    df["JER_without_LM"] = jers
    df["Purity_without_LM"] = purities
    df["Coverage_without_LM"] = coverages

    return df


## Initialize models

In [15]:
# torch.cuda.empty_cache()
import torch, gc

# gc.collect()
# torch.cuda.empty_cache()

diarizer = PyannoteProcessor()
asr = WhisperProcessor()
audio_processor = AudioProcessor(diarizer, asr)

llm_local = None
if RUN_LOCAL_LLM:
    llm_local = LocalLLMPostProcessor(LOCAL_LLM_MODEL_NAME)
    print(f"Using local LLM: {LOCAL_LLM_MODEL_NAME}")
else:
    print("RUN_LOCAL_LLM=False; running Pyannote + Whisper only.")


RUN_LOCAL_LLM=False; running Pyannote + Whisper only.


## Inspect input

In [16]:
df_input = pd.read_csv(INPUT_CSV)
print(df_input.columns.tolist())
print(df_input.head())

if "text_reference" in df_input.columns:
    print(df_input["text_reference"].iloc[0])

if "speaker_segments_reference" in df_input.columns:
    print(df_input["speaker_segments_reference"].iloc[0])

if "text_reference" not in df_input.columns:
    print("Warning: text_reference not found. WER and cpWER will be NaN.")

if "speaker_segments_reference" not in df_input.columns:
    print("Warning: speaker_segments_reference not found. DER/JER/Purity/Coverage will be NaN.")

['audio_number', 'meeting_id', 'audio_condition', 'audio_path', 'text_reference', 'speaker_segments_reference', 'num_speakers', 'num_reference_segments', 'num_reference_words', 'reference_start', 'reference_end', 'source_audio_path']
        audio_number meeting_id     audio_condition  \
0  EN2001a.Headset-0    EN2001a  individual_headset   
1  EN2001a.Headset-1    EN2001a  individual_headset   
2  EN2001a.Headset-2    EN2001a  individual_headset   
3  EN2001a.Headset-3    EN2001a  individual_headset   
4  EN2001a.Headset-4    EN2001a  individual_headset   

                                          audio_path  \
0  /media/user/New Volume/datasets/AMI/amicorpus/...   
1  /media/user/New Volume/datasets/AMI/amicorpus/...   
2  /media/user/New Volume/datasets/AMI/amicorpus/...   
3  /media/user/New Volume/datasets/AMI/amicorpus/...   
4  /media/user/New Volume/datasets/AMI/amicorpus/...   

                                      text_reference  \
0  SPEAKER_00: 'Kay\nSPEAKER_01: Okay\nSPE

In [17]:
required_columns = ["audio_number", "audio_path"]
missing = [c for c in required_columns if c not in df_input.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

In [18]:
optional_columns = ["text_reference", "speaker_segments_reference"]
print("Optional columns found:", [c for c in optional_columns if c in df_input.columns])

Optional columns found: ['text_reference', 'speaker_segments_reference']


## Run pipeline and save raw outputs

In [19]:
df_input = pd.read_csv(INPUT_CSV)

if RUN_LOCAL_LLM:
    llm_variants = {
        f"{LOCAL_LLM_VARIANT_PREFIX}_zero_shot": (llm_local, "zero_shot"),
        f"{LOCAL_LLM_VARIANT_PREFIX}_one_shot": (llm_local, "one_shot"),
    }
else:
    llm_variants = {}

df_raw = run_pipeline_on_dataset(
    df=df_input,
    audio_processor=audio_processor,
    llm_variants=llm_variants,
    output_csv=RAW_OUTPUT_CSV,
    resume=True,
)
print(df_raw.head())
print(f"Saved raw outputs to: {RAW_OUTPUT_CSV}")


  0%|          | 0/575 [00:00<?, ?it/s]/home/user/anaconda3/envs/fgcs/lib/python3.12/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/home/user/anaconda3/envs/fgcs/lib/python3.12/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)
  1%|          | 5/575 [47:56<91:05:39, 575.33s/it]


KeyboardInterrupt: 

## Evaluate and save metrics

In [ ]:
df_eval = pd.read_csv(RAW_OUTPUT_CSV)
df_eval = evaluate_all_methods(df_eval)
df_eval.to_csv(METRICS_OUTPUT_CSV, index=False)

print(df_eval.head())
print(f"Saved metrics to: {METRICS_OUTPUT_CSV}")

## Text metrics summary

In [ ]:
df = pd.read_csv(METRICS_OUTPUT_CSV)

text_metrics = ["WER", "cpWER"]
methods = [column.replace("WER_", "") for column in df.columns if column.startswith("WER_")]

summary_text = {}
for metric in text_metrics:
    summary_text[metric] = [df[f"{metric}_{method}"].mean() for method in methods]

summary_text_df = pd.DataFrame(summary_text, index=methods)
print(summary_text_df)

summary_text_df.to_csv(SUMMARY_TEXT_CSV)
print(f"Saved text summary to: {SUMMARY_TEXT_CSV}")


## Speaker metrics summary

In [ ]:
df = pd.read_csv(METRICS_OUTPUT_CSV)

speaker_summary = pd.DataFrame([{
    "DER_without_LM": df["DER_without_LM"].mean(),
    "JER_without_LM": df["JER_without_LM"].mean(),
    "Purity_without_LM": df["Purity_without_LM"].mean(),
    "Coverage_without_LM": df["Coverage_without_LM"].mean(),
}])

print(speaker_summary)

speaker_summary.to_csv(SUMMARY_SPEAKER_CSV, index=False)
print(f"Saved speaker summary to: {SUMMARY_SPEAKER_CSV}")

# Examples

Best and worst cpWER examples

In [ ]:
df = pd.read_csv(METRICS_OUTPUT_CSV)

cpWER_columns = [
    column for column in df.columns
    if column.startswith("cpWER_")
]

if not cpWER_columns:
    print("No cpWER columns found.")
else:
    min_cpWER = df[cpWER_columns].min().min()
    max_cpWER = df[cpWER_columns].max().max()
    min_cpWER_column = df[cpWER_columns].min().idxmin()
    max_cpWER_column = df[cpWER_columns].max().idxmax()

    min_cpWER_row = df[df[min_cpWER_column] == min_cpWER]
    max_cpWER_row = df[df[max_cpWER_column] == max_cpWER]

    print("Best cpWER example:")
    print(min_cpWER_row[[
        "audio_number",
        "transcript_without_LM",
        min_cpWER_column,
    ]])

    print("\nWorst cpWER example:")
    print(max_cpWER_row[[
        "audio_number",
        "transcript_without_LM",
        max_cpWER_column,
    ]])


In [ ]:
elapsed_time = (time.time() - start_time) / 60
print(f"{elapsed_time:.2f} minutes")

## Release GPU before LLM


In [ ]:
import gc
import torch

for name in ["diarizer", "asr", "audio_processor", "llm_local"]:
    if name in globals():
        del globals()[name]

# gc.collect()
# torch.cuda.empty_cache()

print(torch.cuda.memory_summary()[:1000])


## LLM-only post-processing


In [ ]:
LLM_RAW_OUTPUT_CSV = f"{WORK_ROOT}/raw_pipeline_outputs_{RUN_NAME}_with_llm_chunked.csv"
LLM_METRICS_OUTPUT_CSV = f"{WORK_ROOT}/results_with_metrics_{RUN_NAME}_with_llm_chunked.csv"
LLM_SUMMARY_TEXT_CSV = f"{WORK_ROOT}/summary_text_metrics_{RUN_NAME}_with_llm_chunked.csv"

llm_local = LocalLLMPostProcessor(LOCAL_LLM_MODEL_NAME)

df_llm = pd.read_csv(RAW_OUTPUT_CSV)

for idx, row in tqdm(df_llm.iterrows(), total=len(df_llm)):
    if row.get("status") == "failed":
        continue

    baseline = row["transcript_without_LM"]

    for shot_type in ["zero_shot", "one_shot"]:
        variant_name = f"{LOCAL_LLM_VARIANT_PREFIX}_{shot_type}_chunked"

        try:
            corrected = correct_transcript_in_chunks(
                llm_model=llm_local,
                transcript=baseline,
                shot_type=shot_type,
                max_lines=40,
            )

            if not is_valid_speaker_transcript(corrected):
                print(f"[WARN] Invalid LLM output for {row['audio_number']} {shot_type}. Keeping baseline.")
                corrected = baseline

            if not lexical_content_preserved(baseline, corrected):
                print(f"[WARN] Full lexical content changed for {row['audio_number']} {shot_type}. Keeping baseline.")
                corrected = baseline

            df_llm.loc[idx, f"transcript_{variant_name}"] = corrected

            print(f"{variant_name} same as baseline:", corrected == baseline)

        except Exception as e:
            print(f"[ERROR] LLM failed for {row['audio_number']} {shot_type}: {e}")
            df_llm.loc[idx, f"transcript_{variant_name}"] = baseline

df_llm.to_csv(LLM_RAW_OUTPUT_CSV, index=False)
print(f"Saved LLM raw outputs to: {LLM_RAW_OUTPUT_CSV}")


## Evaluate LLM output


In [ ]:
df_eval_llm = pd.read_csv(LLM_RAW_OUTPUT_CSV)
df_eval_llm = evaluate_all_methods(df_eval_llm)
df_eval_llm.to_csv(LLM_METRICS_OUTPUT_CSV, index=False)

metric_cols = [
    c for c in df_eval_llm.columns
    if c.startswith("WER_") or c.startswith("cpWER_")
]

print(df_eval_llm[["audio_number"] + metric_cols + ["DER_without_LM", "JER_without_LM"]])

methods = [column.replace("WER_", "") for column in df_eval_llm.columns if column.startswith("WER_")]

summary_text_df = pd.DataFrame({
    "WER": [df_eval_llm[f"WER_{method}"].mean() for method in methods],
    "cpWER": [df_eval_llm[f"cpWER_{method}"].mean() for method in methods],
}, index=methods)

print(summary_text_df)
summary_text_df.to_csv(LLM_SUMMARY_TEXT_CSV)
print(f"Saved LLM text summary to: {LLM_SUMMARY_TEXT_CSV}")


## LLM text metrics summary


In [ ]:
df = pd.read_csv(LLM_METRICS_OUTPUT_CSV)

methods = [column.replace("WER_", "") for column in df.columns if column.startswith("WER_")]

summary_text_df = pd.DataFrame({
    "WER": [df[f"WER_{method}"].mean() for method in methods],
    "cpWER": [df[f"cpWER_{method}"].mean() for method in methods],
}, index=methods)

print(summary_text_df)
summary_text_df.to_csv(LLM_SUMMARY_TEXT_CSV)
print(f"Saved LLM text summary to: {LLM_SUMMARY_TEXT_CSV}")


In [ ]:
df = pd.read_csv(LLM_RAW_OUTPUT_CSV)

for col in df.columns:
    if col.startswith("transcript_"):
        print(col)
        print("same as baseline:", df[col].iloc[0] == df["transcript_without_LM"].iloc[0])
        print("length:", len(df[col].iloc[0]))
        print()